This notebook examines the performance of different hypothesis testing methods using shendure-calibrated simulated datasets

Imports

In [1]:
%load_ext autoreload
%autoreload 2
import scMPRAforge as scm

2025-10-16 11:54:11.802477: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-16 11:54:11.833722: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /vast/palmer/apps/avx2/software/Code-Server/4.17.0/lib:/vast/palmer/apps/avx2/software/gettext/0.22.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libiconv/1.17-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/ncurses/6.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/XZ/5.4.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/expat/2.6.2-GCCcore-13.3.0/lib:/vast/palmer/apps/av

Cluster setup

In [2]:
#create dask cluster
from dask.distributed import Client, LocalCluster
from dask_jobqueue import SLURMCluster

cluster=SLURMCluster(
    cores=2,#cores per slurm job
    memory="16G",#memory per slurm job
    processes=1,#dask workers per slurm job
    local_directory="/tmp",
    job_extra_directives=["-p ycga", 
        f"--job-name=clust_worker",
        f"--time=1:00:00",
        f"--output=slave_%j.out"]
)

cluster.scale(jobs=2)

client = Client(cluster,
        timeout=f"{5*60}s",   # Client <-> scheduler timeout 
        heartbeat_interval="20s"  # Worker heartbeat interval
    )


First, we load the simulations w/ orthos:

In [3]:
multisim=scm.de_novo_simulation.load(client,path="/gpfs/gibbs/pi/reilly/tabula_data/simulated/",name="shendure_calibrated_sim_with_orthos_20251008")

In [4]:
multisim.orthos

Next, we create some hypothesis sets...

In [5]:
#grab one of the simulated datasets to get some  
example_counts=multisim.simulated_scMPRA[0].result()

In [6]:
# all CREs within each cell type, vs the 'reference' negative control
hs_all_ct = scm.make_all_by_celltype_hypotheses(
    counts=example_counts,
    reference_cre="reference",
    meta="emvar_screen",
)

# all cell types for each CRE, vs the dataset’s baseline cell type
hs_all_cre = scm.make_all_by_cre_hypotheses(
    counts=example_counts,
    reference_cell_type="reference",  # will be normalized to 'reference'
    meta="cell_specificity",
)

Now let's run the wald...

In [7]:
multisim._test_replicate(client,hs_all_ct,test="wald",index=0)
#multisim._test_all_replicates(client,hs_all_ct,test="wald")

: 

Shut down the cluster

In [ ]:
client.close()
cluster.close()